In [1]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

from budget import load_budget, extract_cashflow
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
budget = load_budget()
income, expenses = extract_cashflow(budget["incomes"]), extract_cashflow(
    budget["expenses"]
)

In [3]:
OPERATION_CASH = 23080.09

In [4]:
budget_with_balance = (
    pl.concat(
        [income.with_columns(is_income=True), expenses.with_columns(is_income=False)]
    )
    .sort("date")
    .with_columns(net_amount=pl.col("amount") * (pl.col("is_income") * 2 - 1))
    .with_columns(
        balance=(-pl.col("net_amount")).cum_sum(reverse=True) + OPERATION_CASH
    )
)

In [5]:
aggregated_budget_with_balance = budget_with_balance.group_by(
    date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
).agg(pl.col("balance").last(), pl.col("net_amount").sum()).sort('date')

In [6]:
balance_fig = (
    make_subplots(
        rows=2,
        cols=1,
        subplot_titles=("Monthly Aggregated Balance", "Balance by Transaction"),
    )
    .add_trace(
        go.Scatter(
            x=aggregated_budget_with_balance["date"],
            y=aggregated_budget_with_balance["balance"],
            name="balance",
        ),
        row=1,
        col=1,
    )
    .add_trace(
        go.Bar(
            x=aggregated_budget_with_balance["date"],
            y=aggregated_budget_with_balance["net_amount"],
            name="transactions",
        ),
        row=1,
        col=1,
    )
    .add_trace(
        go.Scatter(
            x=budget_with_balance["date"],
            y=budget_with_balance["balance"],
            name="balance",
        ),
        row=2,
        col=1,
    )
    .add_trace(
        go.Bar(
            x=budget_with_balance["date"],
            y=budget_with_balance["net_amount"],
            name="transaction",
            customdata=budget_with_balance.select(
                "headline_category", "category", "description"
            ),
            hovertemplate="%{x}<br>amount=%{y}<br>headline category=%{customdata[0]}<br>category=%{customdata[1]}<br>description=%{customdata[2]}",
        ),
        row=2,
        col=1,
    )
    .update_layout(width=1500, height=1000)
)
balance_fig.show()

In [7]:
balance_fig.write_html('balance.html')